## **Setup & Load Data**

In [ ]:
# install.packages("FNN")
# install.packages("ggrepel")
library(FNN)
library(lubridate)
library(tidyverse)
library(readr)
library(stringr)
library(bigrquery)
library(parallel)
library(ggrepel)

In [ ]:
EXPORT_BUCKET = "gs://plm-pers-hba1c-thresh-wb-silky-pepper-6055"
BILLING = "wb-silky-pepper-6055"
DATA_MOUNT = "/home/jupyter/workspace/raw/vwb-aou-datasets-controlled/v8"
# CDR_STORAGE_PATH = "gs://fc-aou-datasets-controlled/v8"
# WORKSPACE_CDR = "wb-silky-artichoke-2408.C2024Q3R8"
# previous proj = "terra-vpc-sc-d3cc1fbe"

In [ ]:
# Get analysis data
system(paste0("gsutil cp ", EXPORT_BUCKET, "/data_analysis_tsh.txt ./"))
data_anal <- read_tsv("data_analysis_tsh.txt")

# Get matched cohorts
system(paste0("gsutil cp ", EXPORT_BUCKET, "/matched_cohorts_tsh.txt ./"))
knn_cohort_t <- read_tsv("matched_cohorts_tsh.txt")

# Get personalized shifts
system(paste0("gsutil cp ", EXPORT_BUCKET, "/tsh_anal_shift.txt ./"))
df <- read.table("tsh_anal_shift.txt", header = TRUE)

## **Analysis**

In [ ]:
# Definitions re-used throughout
tsh_low <- 0.4
tsh_high <- 4.0

#### **Measurement-Level Confusion Matrix**

TO NOTE: this preliminary version is based only on measurement thresholds, does not include ICD codes.

In [ ]:
# Think will first do this for all measures for every individual
# 1. classify each measure based on standard thresholds
# 2. classify each measure based on personalized thresholds
df_mod <- df %>%
    dplyr::mutate(
        id = as.character(format(id, scientific = FALSE, trim = TRUE))) %>%
    dplyr::mutate(
        pers_tsh_low = tsh_low - shift,
        pers_tsh_high = tsh_high - shift)

data_anal_thresh <- inner_join(
    data_anal %>% dplyr::mutate(person_id = as.character(person_id)),
    df_mod,
    by = c("person_id" = "id"))

In [ ]:
dim(data_anal_thresh)
head(data_anal_thresh)

In [ ]:
# Make confusion matrix
data_anal_thresh <- data_anal_thresh %>%
    dplyr::mutate(conf_low = case_when(
        tsh > tsh_low & tsh > pers_tsh_low ~ "tn",
        tsh > tsh_low & tsh <= pers_tsh_low ~ "fn",
        tsh <= tsh_low & tsh > pers_tsh_low ~ "fp",
        tsh <= tsh_low & tsh <= pers_tsh_low ~ "tp"
    )) %>%
    dplyr::mutate(conf_high = case_when(
        tsh < tsh_high & tsh < pers_tsh_high ~ "tn",
        tsh < tsh_high & tsh >= pers_tsh_high ~ "fn",
        tsh >= tsh_high & tsh < pers_tsh_high ~ "fp",
        tsh >= tsh_high & tsh >= pers_tsh_high ~ "tp"
    ))

In [ ]:
table(data_anal_thresh$conf_low)
table(data_anal_thresh$conf_high)

#### **Updated Individual-Level Summary**

Define true early warning: low/high personalized before standard OR never standard

Need to get each person's earliest date of:

+ low TSH
+ high TSH
+ pers low TSH
+ pers high TSH
+ normal?

In [ ]:
# Modifying code from Hayley
# Note that this removes individuals with only "normal" measurements
date_summ <- data_anal_thresh %>%
  dplyr::mutate(
    pers_low = tsh <= pers_tsh_low,
    classic_low = tsh <= tsh_low,
    pers_high = tsh >= pers_tsh_high,
    classic_high = tsh >= tsh_high
  ) %>%
  tidyr::pivot_longer(
    cols = c(pers_low, classic_low, pers_high, classic_high),
    names_to = c("type", "direction"),
    names_sep = "_",
    values_to = "flag"
  ) %>%
  dplyr::filter(flag) %>%
  dplyr::group_by(person_id, type, direction) %>%
  dplyr::summarise(first_date = min(datetime), .groups = "drop") %>%
  dplyr::mutate(date_col = paste0("tsh_", direction, "_date_", type)) %>%
  dplyr::select(person_id, date_col, first_date) %>%
  tidyr::pivot_wider(
    names_from = date_col,
    values_from = first_date)

In [ ]:
head(date_summ)

In [ ]:
# Summary
cat("Distinct individuals:", n_distinct(data_anal_thresh$person_id), "\n")
cat("Distinct individuals with only normal classic/personalized:", length(setdiff(unique(data_anal_thresh$person_id), unique(date_summ$person_id))), "\n")
cat("Distinct individuals with classic/personalized high/low TSH:", n_distinct(date_summ$person_id), "\n")

In [ ]:
indv_low <- date_summ %>%
    dplyr::filter(!(is.na(tsh_low_date_pers) & is.na(tsh_low_date_classic))) %>%
    dplyr::mutate(
        low_status = dplyr::case_when(
            !is.na(tsh_low_date_pers) & is.na(tsh_low_date_classic) ~ "pers_only",
            
            !is.na(tsh_low_date_pers) & !is.na(tsh_low_date_classic) &
                tsh_low_date_pers < tsh_low_date_classic ~ "pers_early",
            TRUE ~ "no_diff"))
dim(indv_low)

In [ ]:
indv_high <- date_summ %>%
    dplyr::filter(!(is.na(tsh_high_date_pers) & is.na(tsh_high_date_classic))) %>%
    dplyr::mutate(
        high_status = dplyr::case_when(
            !is.na(tsh_high_date_pers) & is.na(tsh_high_date_classic) ~ "pers_only",
            
            !is.na(tsh_high_date_pers) & !is.na(tsh_high_date_classic) &
                tsh_high_date_pers < tsh_high_date_classic ~ "pers_early",
            TRUE ~ "no_diff"))
dim(indv_high)

In [ ]:
# For now, not worrying about classifying the no difference further
indv_low %>%
  count(low_status) %>%
  mutate(percent = (n / sum(n))*100)

indv_high %>%
  count(high_status) %>%
  mutate(percent = (n / sum(n))*100)

In [ ]:
hba1c_anal_thresh %>% dplyr::filter(person_id == "1000004") %>%
    dplyr::arrange(datetime)

head(indv_prediab)

#### **Updated Individual-Level Summary - With Exclusions**

Exclude individuals where first measurement > 6.5 for all analyses.
Exclude individuals where first measurement > 5.7 for prediabetes analyses.

In [ ]:
# For now, just excluding individuals from the non-normal subset
#TODO: Later, revisit and make exlusion earlier if want to keep
#TODO: Later revisit and see if catching technical outliers and not truly
# high first measurements?
id_list_filt <- hba1c_anal_thresh %>%
  arrange(person_id, datetime) %>%
  group_by(person_id) %>%
  filter(first(hba1c) <= diab) %>%
  ungroup() %>%
  dplyr::pull(person_id)

n_distinct(id_list_filt)

In [ ]:
id_list_filt_prediab <- hba1c_anal_thresh %>%
  arrange(person_id, datetime) %>%
  group_by(person_id) %>%
  filter(first(hba1c) <= prediab) %>%
  ungroup() %>%
  dplyr::pull(person_id)

n_distinct(id_list_filt_prediab)

In [ ]:
early_warning_summ_diab_filt <- early_warning_summ %>%
    dplyr::filter(person_id %in% id_list_filt)
dim(early_warning_summ_diab_filt)

early_warning_summ_prediab_filt <- early_warning_summ %>%
    dplyr::filter(person_id %in% id_list_filt_prediab)
dim(early_warning_summ_prediab_filt)

In [ ]:
ct_prediab_filt_true_early <- sum(early_warning_summ_prediab_filt$prediab_true_early)
print(ct_prediab_filt_true_early)
ct_diab_filt_true_early <- sum(early_warning_summ_diab_filt$diab_true_early)
print(ct_diab_filt_true_early)

cat("Percentage prediab true early warning:", (ct_prediab_filt_true_early / n_distinct(early_warning_summ_prediab_filt$person_id)) * 100, "%\n")
cat("Percentage diab true early warning:", (ct_diab_filt_true_early / n_distinct(early_warning_summ_diab_filt$person_id)) * 100, "%\n")

#### **Example Individual Plot like CCPM**

TO NOTE: running for grant submission without any additional exclusions.

TO NOTE: current code is set up to exactly match previous analysis by Matthew Joel
in CCPM, but should be revisited due to geom_density behavior with scale limits.

In [ ]:
# Find good example
    # less than 10 measurements
    # true early warning
    # shift > 0.2?
early_warning_summ_list <- early_warning_summ %>%
    dplyr::filter(prediab_true_early == TRUE & diab_true_early == TRUE) %>%
    dplyr::pull(person_id)

plot_df <- hba1c_anal_thresh %>%
    dplyr::filter(person_id %in% early_warning_summ_list) %>%
    dplyr::filter(shift > 0.2) %>%
    dplyr::group_by(person_id) %>%
    dplyr::filter(n() < 10) %>%
    dplyr::arrange(datetime, .by_group = TRUE) %>%
    dplyr::mutate(measure_order = paste0("v", row_number())) %>%
    dplyr::ungroup()

In [ ]:
print(plot_df)

In [ ]:
id <- "2072208"
cohort_ids <- unlist(knn_cohort_t[, id], use.names = F)

group_data <- hba1c_anal_thresh %>%
    dplyr::filter(person_id %in% cohort_ids)

pers_prediab <- unique(hba1c_anal_thresh$pers_prediab[hba1c_anal_thresh$person_id == id])
pers_diab <- unique(hba1c_anal_thresh$pers_diab[hba1c_anal_thresh$person_id == id])
shift_val <- unique(hba1c_anal_thresh$shift[hba1c_anal_thresh$person_id == id])

# full_d <- density(hba1c_anal$hba1c, na.rm = TRUE)
# full_d_peak <- full_d$x[which.max(full_d$y)]
cohort_d <- density(group_data$hba1c, na.rm = TRUE)
cohort_d_peak <- cohort_d$x[which.max(cohort_d$y)]

In [ ]:
#TODO: after grant, may want to plot both global and cohort with same bandwidth,
# but leaving as is for now for comparison with CCPM

# Threshold colors
col_pre <- "#B8860B" # Dark Gold
col_diab <- "#8B0000" # Dark Red

# Prep threshold lines
vlines <- data.frame(
  x = c(prediab, diab, pers_prediab, pers_diab),
  threshold = c("Classic Pre", "Classic Diab", "Cohort Pre", "Cohort Diab"))

options(repr.plot.width = 12, repr.plot.height = 6)
p_anno <- ggplot() +
    # Add density curves
    geom_density(
      data = hba1c_anal_thresh,
      aes(x = hba1c, fill = "All"),
      adjust = 2,
      alpha = 0.5,
      color = NA) +
    geom_density(
      data = group_data,
      aes(x = hba1c, fill = "Cohort"),
      alpha = 0.5,
      color = NA) +
    # Add separate outlines off legend
    geom_density(
      data = hba1c_anal_thresh,
      aes(x = hba1c),
      adjust = 2,
      color = "grey50",
      size = 0.7,
      fill = NA,
      show.legend = FALSE) +
    geom_density(
      data = group_data,
      aes(x = hba1c),
      color = "steelblue",
      size = 0.7,
      fill = NA,
      show.legend = FALSE) +
    # Add measurement points and labels
    geom_vline(
      data = vlines,
      aes(xintercept = x, color = threshold, linetype = threshold), 
      linewidth = 0.7,
      key_glyph = draw_key_path) +
    geom_point(
      data = hba1c_anal_thresh %>% dplyr::filter(person_id == id),
      aes(x = hba1c, y = 0),
      color = "black",
      fill = "black",
      size = 3,
      stroke = 0.3) +
    ggrepel::geom_text_repel(
      data = plot_df %>% dplyr::filter(person_id == id),
      aes(x = hba1c, y = 0, label = measure_order),
      nudge_y = 0.03,
      size = 3,
      segment.colour = "grey60",
      max.overlaps = Inf,
      show.legend = FALSE) +
    scale_fill_manual(
        name = "Distribution",
        values = c("All" = "grey70", "Cohort" = "steelblue")) +
    scale_color_manual(
        name = "Thresholds",
        breaks = c("Classic Pre", "Classic Diab", "Cohort Pre", "Cohort Diab"),
        values = c(
          "Classic Pre" = col_pre,
          "Classic Diab" = col_diab,
          "Cohort Pre" = col_pre,
          "Cohort Diab" = col_diab),
    labels = c(
      paste0("Classic Pre ", round(prediab, 2), "%"),
      paste0("Classic Diab ", round(diab, 2), "%"),
      paste0("Cohort Pre ", round(pers_prediab, 2), "%"),
      paste0("Cohort Diab ", round(pers_diab, 2), "%")
    )
  ) +
  scale_linetype_manual(
    name = "Thresholds",
    breaks = c("Classic Pre", "Classic Diab", "Cohort Pre", "Cohort Diab"),
    values = c(
      "Classic Pre" = "solid",
      "Classic Diab" = "solid",
      "Cohort Pre" = "dashed",
      "Cohort Diab" = "dashed"
    ),
    labels = c(
      paste0("Classic Pre ", round(prediab, 2), "%"),
      paste0("Classic Diab ", round(diab, 2), "%"),
      paste0("Cohort Pre ", round(pers_prediab, 2), "%"),
      paste0("Cohort Diab ", round(pers_diab, 2), "%")
    )
  ) +
  scale_x_continuous(
      limits = c(3.5, 7.5),
      breaks = seq(3.5, 7.5, 0.5)) +
  scale_y_continuous(
      limits = c(0, 0.90)) +
  labs(
    title = "HbA1c Density: Global vs. Left-Shifted Cohort",
    subtitle = sprintf(
      "Shift: %.2f%% (Global Peak %.2f - Cohort Peak %.2f)",
      shift_val, full_d_peak, cohort_d_peak
    ),
    x = "HbA1c (%)",
    y = "Density"
  ) +
  theme_minimal(base_size = 13) +
  theme(
    legend.position = "right",
    legend.box = "vertical",
    legend.key.width = unit(1.5, "cm"),
    legend.key.height = unit(0.4, "cm"))

In [ ]:
# Show annotated plot
p_anno

In [ ]:
# Show simplified plot for grant
p_simp <- p_anno +
    labs(title = NULL, subtitle = NULL) +
    theme(
        legend.position = "none")
p_simp

In [ ]:
ggsave(
    "aou_example_trajectory_anno.png",
    plot = p_anno,
    width = 9, height = 4.5, units = "in",
    dpi = 600, device = ragg::agg_png, bg = "white")

ggsave(
    "aou_example_trajectory.png",
    plot = p_simp, width = 7, height = 4.2, units = "in",
    dpi = 600, device = ragg::agg_png, bg = "white")

#### **Percentage Left- & Right-Shifted**

In [ ]:
shift_summ <- df_mod %>%
    dplyr::select(id, shift) %>%
    dplyr::filter(!duplicated(.)) %>%
    dplyr::mutate(shift_dir = case_when(
        shift < 0 ~ "neg",
        shift > 0 ~ "pos", 
        shift == 0 ~ "none"))
dim(shift_summ)

In [ ]:
table(shift_summ$shift_dir)

In [ ]:
# positive shift means matched cohort density shifted to lower TSH
shift_summ_pos <- shift_summ %>%
    dplyr::filter(shift_dir == "pos")
summary(shift_summ_pos$shift)

In [ ]:
# positive shift means matched cohort density shifted to lower TSH
shift_summ_neg <- shift_summ %>%
    dplyr::filter(shift_dir == "neg")
summary(shift_summ_neg$shift)

In [ ]:
# Sanity check example ID from above in this
"2072208" %in% shift_summ_pos$id
data_anal_thresh %>% dplyr::filter(person_id == "2072208")